In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
ROOT

PosixPath('/Users/v_angelov/GroupProj/TP53-MUTATIONS')

# 1. Download TCGA Data

In [2]:
# I had already downloaded the data, so I am skipping this step. If you want to download the data, please run the following command:
!python scripts/00_download_tcga.py 

[skip] tcga_expression.tsv.gz already exists
[skip] tcga_mutations.tsv.gz already exists
[skip] tcga_phenotype.tsv.gz already exists

All TCGA files downloaded.


# 2. Build Dataset

In [3]:
!python scripts/01_build_tcga_dataset.py 

Loading phenotype …
  Primary tumor samples: 10,593
Loading expression matrix (this may take ~30 s) …
  Expression after primary-tumor filter: (9701, 20531)
  Filling 3,197,038 NaN values with 0 (undetected genes)
  Dropped 29 numeric-ID columns and 220 zero-variance genes; kept 20,281 gene symbols
Building TP53 labels …
  Collapsing into 'Other' (< 100 samples): ['In-frame indel', 'Other', 'Synonymous']
  TP53 mutant: 3,130
  TP53 wild-type: 6,571
  Mutation type counts:
mutation_type_collapsed
WT            6571
Missense      1949
Nonsense       444
Frameshift     401
Splice         215
Other          121
Saving processed files …

Done. Files written to data/processed/tcga/
  expression_matched.csv.gz : (9701, 20281)
  tp53_labels.csv           : (9701, 4)
  sample_metadata.csv       : (9701, 3)


# 3. Train binary classifier

In [4]:
!python scripts/03_train_binary.py --data-dir data/processed/tcga --tag tcga

Minimum samples required per stratum: 7
Samples before filtering: 9,701
Samples after filtering:  8,113
Samples removed:          1,588
Cancer types retained:    23
Strata retained:          91
top_3000_variable / majority: val ROC-AUC=0.500, F1=0.000
top_3000_variable / logistic_l2: val ROC-AUC=0.867, F1=0.735
/Users/v_angelov/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
top_3000_variable / elastic_net_logistic: val ROC-AUC=0.889, F1=0.774
top_3000_variable / linear_svm: val ROC-AUC=0.853, F1=0.711
top_3000_variable / random_forest: val ROC-AUC=0.886, F1=0.720
top_3000_variable / extra_trees: val ROC-AUC=0.887, F1=0.718
top_3000_v

# 4. Train multiclass model — TCGA (Splice as own class)

In [ ]:
#!python scripts/04_train_multiclass.py --data-dir data/processed/tcga --tag tcga

# 5. Train multiclass model — TCGA (Splice merged into Other)

CCLE already collapses Splice → Other in its label builder, so this run makes the two datasets directly comparable.

In [5]:
!python scripts/04_train_multiclass.py --data-dir data/processed/tcga --tag tcga_nosplice --merge-splice

Minimum samples required per cancer type: 7
Samples before cancer-type filter: 9,701
Samples after cancer-type filter:  9,701
Cancer types retained: 32
Samples after mutation-class filter: 9,701
Classes retained: ['Frameshift', 'Missense', 'Nonsense', 'Other', 'WT']
top_3000_variable / majority: val macro-F1=0.162
top_3000_variable / logistic_l2: val macro-F1=0.337
top_3000_variable / elastic_net_logistic: val macro-F1=0.317
top_3000_variable / linear_svm: val macro-F1=0.297
top_3000_variable / random_forest: val macro-F1=0.267
top_3000_variable / extra_trees: val macro-F1=0.278
top_3000_variable / hist_gradient_boosting: val macro-F1=0.297
top_3000_variable / mlp: val macro-F1=0.354
tp53_targets / majority: val macro-F1=0.162
tp53_targets / logistic_l2: val macro-F1=0.405
tp53_targets / elastic_net_logistic: val macro-F1=0.407
tp53_targets / linear_svm: val macro-F1=0.423
tp53_targets / random_forest: val macro-F1=0.400
tp53_targets / extra_trees: val macro-F1=0.410
tp53_targets / his